In [ ]:
!pip install -q transformers accelerate sentencepiece opencv-python-headless pillow

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from google.colab import files


MODEL_NAME = "microsoft/trocr-base-handwritten"


device = "cuda" if torch.cuda.is_available() else "cpu"

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Loaded:", MODEL_NAME, "| device:", device)

def segment_lines(img_path, pad=4):
    bgr = cv2.imread(img_path)
    if bgr is None:
        raise FileNotFoundError(img_path)

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    _, bw = cv2.threshold(
        gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 1))
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, k)

    proj = bw.sum(axis=1)
    thr = proj.max() * 0.05
    rows = proj > thr

    regions = []
    in_line = False
    start = 0

    for i, r in enumerate(rows):
        if r and not in_line:
            in_line = True
            start = i
        elif not r and in_line:
            in_line = False
            if i - start > 8:
                regions.append((max(0, start - pad), min(bgr.shape[0], i + pad)))

    if in_line and bgr.shape[0] - start > 8:
        regions.append((max(0, start - pad), bgr.shape[0]))

    pil = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    lines = [pil.crop((0, y1, pil.width, y2)) for y1, y2 in regions]

    return lines, bgr, regions

def tight_crop_line(pil_img, margin=3):
    arr = np.array(pil_img.convert("L"))
    ink = 255 - arr
    col_sum = ink.sum(axis=0)

    if col_sum.max() <= 0:
        return pil_img

    cols = np.where(col_sum > 0.02 * col_sum.max())[0]
    if len(cols) == 0:
        return pil_img

    x1 = max(0, int(cols[0]) - margin)
    x2 = min(arr.shape[1], int(cols[-1]) + margin + 1)

    return pil_img.crop((x1, 0, x2, arr.shape[0]))


def make_line_variants(pil_img):
    img = tight_crop_line(pil_img).convert("RGB")
    gray = ImageOps.grayscale(img)

    variants = [
        ("orig", img),
        ("autocontrast", ImageOps.autocontrast(img)),
        ("gray", gray.convert("RGB")),
        ("gray_autocontrast", ImageOps.autocontrast(gray).convert("RGB")),
        ("sharpen", img.filter(ImageFilter.SHARPEN)),
    ]

    return variants


@torch.no_grad()
def predict_line_trocr_best(pil_img, max_new_tokens=128, show_variants=False):
    candidates = []

    for name, var_img in make_line_variants(pil_img):
        pixel_values = processor(images=var_img, return_tensors="pt").pixel_values.to(device)

        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=max_new_tokens,
            num_beams=6,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

        score = len([c for c in text if c.isalnum() or c in " .,!?;:'\"-()"])
        candidates.append((score, name, text))

    candidates.sort(reverse=True)

    if show_variants:
        print("Top variants:")
        for score, name, text in candidates:
            print(f"{name:18s} | score={score:3d} | {text}")

    best_score, best_name, best_text = candidates[0]
    return best_text, best_name, candidates


def recognize_page_trocr(img_path, show=True, show_variants=False):
    lines, dbg, regions = segment_lines(img_path)
    print(f"{len(lines)} line detected")

    if not lines:
        print("No line found")
        return ""

    if show:
        dbg2 = dbg.copy()
        for i, (y1, y2) in enumerate(regions):
            cv2.rectangle(dbg2, (0, y1), (dbg.shape[1], y2), (0, 200, 0), 2)
            cv2.putText(
                dbg2,
                str(i + 1),
                (8, y1 + 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 220),
                2
            )

        plt.figure(figsize=(14, 8))
        plt.imshow(cv2.cvtColor(dbg2, cv2.COLOR_BGR2RGB))
        plt.title(f"Detected lines: {len(regions)}")
        plt.axis("off")
        plt.show()

    results = []

    for i, line in enumerate(lines):
        txt, variant_name, candidates = predict_line_trocr_best(
            line,
            show_variants=show_variants
        )
        results.append(txt)

        if show:
            plt.figure(figsize=(14, 1.5))
            plt.imshow(np.array(line))
            plt.title(f'Line {i+1} [{variant_name}]: "{txt}"', fontsize=10, loc="left")
            plt.axis("off")
            plt.tight_layout()
            plt.show()

    full = "\n".join(results)

    print("\n" + "=" * 50)
    print("RESULT:")
    print("=" * 50)
    print(full)

    return full

uploaded = files.upload()
img_path = list(uploaded.keys())[0]

text = recognize_page_trocr(img_path, show=True, show_variants=True)
